In [3]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.store.postgres import PostgresStore
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage
from langgraph.store.base import BaseStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState

import uuid
from typing import List
from pydantic import Field, BaseModel
import os


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [5]:
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP  server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps..." 
say "Since your project is built with TypeScript..."

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [6]:
MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return an empty list.
"""

In [7]:
mem_writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

class MemoryItem(BaseModel):
    text: str = Field(description="Atomic user memory as a short sentence")
    is_new: bool = Field(description="True if this memory is NEW and should be stored. False if duplicate/already known.")

class MemoryDecision(BaseModel):
    should_write: bool = Field(description="Whether to store any memories")
    memories: List[MemoryItem] = Field(default_factory=list, description="Atomic user memories to store")

memory_writer_llm = mem_writer_llm.with_structured_output(MemoryDecision) 

In [8]:
def remember_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    user_id = config['configurable']['user_id']

    namespace = ("users", user_id, "details")

    existing_items = store.search(namespace)
    existing_texts = [it.value.get("data", "") for it in existing_items if it.value.get("data")]
    user_details_content = "\n".join(f"- {t}" for t in existing_texts) if existing_texts else "(empty)"

    # B) Latest user message
    last_text = state["messages"][-1]

    decision = memory_writer_llm.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(user_details_content=user_details_content)),
            {"role": "user", "content": f"USER MESSAGE:\n{last_text}"},
        ]
    )

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new:
                store.put(namespace, str(uuid.uuid4()), {"data": mem.text})

    return {}

In [9]:
chat_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [10]:
def chat_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    
    user_id = config['configurable']['user_id']
    user_details = ("users", user_id, "details")
    items = store.search(user_details)

    if items:
        user_details_content = "\n".join(f"- {it.value.get('data', '')}" for it in items)
    else:
        user_details_content = ""  # prompt says it may be empty

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        user_details_content=user_details_content
    )

    system_message = SystemMessage(
        content=system_prompt
    )

    response = chat_llm.invoke([system_message] + state['messages'])
    return {
        "messages": [response]
    }

In [11]:
builder = StateGraph(MessagesState)

builder.add_node("chat", chat_node)
builder.add_node("remember", remember_node)

builder.add_edge(START, "remember")
builder.add_edge("remember", "chat")
builder.add_edge("chat", END)

In [12]:
DB_URI = os.getenv("DB_URI")

with PostgresStore.from_conn_string(DB_URI) as store:
    # IMPORTANT: run ONCE the first time you use this database
    store.setup()

    graph = builder.compile(store=store)

    config = {"configurable": {"user_id": "u1"}}

    graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ahmad"}]}, config)
    graph.invoke({"messages": [{"role": "user", "content": "I love building AI Agents."}]}, config)

    out = graph.invoke({"messages": [{"role": "user", "content": "Explain GenAI simply"}]}, config)
    print(out["messages"][-1].content[0]['text'])

    print("\n--- Stored Memories (from Postgres) ---")
    for it in store.search(("users", "u1", "details")):
        print(it.value["data"])

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 

Hello Ahmad! It's great to chat with you again. 

Since you love building AI agents, you already have a fantastic intuition for how modern artificial intelligence works. Let’s break down Generative AI (GenAI) in a simple way.

Think of traditional AI as a **critic or a classifier**. It’s like a program trained to look at data and say, *"That’s a cat,"* or *"This email is spam."* It analyzes the world, but it doesn't create anything new.

**Generative AI**, on the other hand, is the **creator**. 

Instead of just sorting data, GenAI is trained on massive amounts of examples—text, code, images, or audio—so it learns the *patterns* behind them. When you prompt it, it uses those patterns to generate brand-new, original content that has never existed before. 

If you ask a GenAI model to write a Python script for a new feature, paint a picture of a cyberpunk city, or draft a conversation flow for one of your AI agents, it doesn't just copy-paste. It predicts what word, pixel, or note should

In [13]:
from langgraph.store.postgres import PostgresStore
import os

DB_URI = os.getenv("DB_URI")

with PostgresStore.from_conn_string(DB_URI) as store:
    ns = ("users", "u1", "details")
    items = store.search(ns)

for it in items:
    print(it.value["data"])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


The user loves building AI Agents.
The user's name is Ahmad.
